# **LassaGuard AI** 
_AI-powered prediction of household Lassa fever risk_


*LassaGuard AI* is a machine learning model that predicts the risk of household-level Lassa fever in Nigeria based on environmental predispositions.

The system can help the government and public health authorities implement targeted prevention strategies and reduce outbreaks in endemic communities by identifying high-risk households early.

### **Synthesizing Model Dataset**

In [1]:
import numpy as np
import pandas as pd

In [9]:
np.random.seed(42)

data = []
for i in range (500):
    row = {
        'season' : np.random.choice(['Dry','Rainy']),
        'environ_vegetation' : np.random.choice(['None','Sparse','Dense']),
        'dump_site_nearby' : np.random.choice(['Yes','No']),
        'heap_of_waste' : np.random.choice(['Yes','No']),
        'waste_disposal_method' : np.random.choice(['Proper','Improper']),
        'toilet_type' : np.random.choice(['Flush','Pit']),
        'toilet_condition' : np.random.choice(['Good','Poor']),
        'shares_toilet' : np.random.choice(['Yes','No']),
        'drainage_system' : np.random.choice(['Good','Poor']),
        'rodent_infestation' : np.random.choice(['Yes','No']),
        'risk_of_lf' : None
    }

    data.append(row)

for row in data:
    score = 0
    if row['rodent_infestation'] == 'Yes':
        score += 3
    if row['environ_vegetation'] == 'Dense':
        score += 1
    if row['dump_site_nearby'] == 'Yes':
        score += 1
    if row['heap_of_waste'] == 'Yes':
        score += 1
    if row['waste_disposal_method'] == 'Improper':
        score += 1
    if row['toilet_type'] == 'Pit':
        score += 1
    if row['toilet_condition'] == 'Poor':
        score += 1
    if row['shares_toilet'] == 'Yes':
        score += 1
    if row['drainage_system'] == 'Poor':
        score += 1
    if row['season'] == 'Dry':
        score += 1
    if score >= 6:
        row['risk_of_lf'] = 'High Risk'
    elif score >= 3:
        row['risk_of_lf'] = 'Moderate Risk'
    else:
        row['risk_of_lf'] = 'Low/No Risk'

df = pd.DataFrame(data)

In [16]:
df.head()

,season,environ_vegetation,dump_site_nearby,heap_of_waste,waste_disposal_method,toilet_type,toilet_condition,shares_toilet,drainage_system,rodent_infestation,risk_of_lf
0,Dry,None,Yes,Yes,Improper,Flush,Good,Yes,Poor,Yes,High Risk
1,Dry,Dense,Yes,No,Proper,Pit,Poor,No,Good,No,Moderate Risk
2,Dry,Sparse,No,No,Improper,Pit,Poor,No,Poor,Yes,High Risk
3,Dry,Sparse,No,Yes,Improper,Flush,Good,Yes,Good,Yes,High Risk
4,Rainy,Dense,No,No,Proper,Pit,Good,No,Good,No,Low/No Risk


In [34]:
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score,auc,roc_curve
import joblib, pickle

import warnings
warnings.filterwarnings('ignore')

### **Feature Engineering and Data Splitting**

In [23]:
target = pd.DataFrame(df['risk_of_lf'])
features = df.drop('risk_of_lf',axis=1)

ord_enc = OrdinalEncoder()
y = ord_enc.fit_transform(target)
one_hot = OneHotEncoder()
x = one_hot.fit_transform(features)

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

### **Model Training and Evaluation**

In [30]:
target_name = ['High Risk','Moderate risk','Low/No Risk']

# Logistic Regression Model

log = LogisticRegression()
log.fit(x_train,y_train)
log_y_pred = log.predict(x_test)
print(classification_report(y_test,log_y_pred,target_names=target_name))

               precision    recall  f1-score   support

    High Risk       1.00      1.00      1.00        66
Moderate risk       1.00      0.50      0.67         4
  Low/No Risk       0.94      1.00      0.97        30

     accuracy                           0.98       100
    macro avg       0.98      0.83      0.88       100
 weighted avg       0.98      0.98      0.98       100



In [31]:
# Decision Tree Model

tree = DecisionTreeClassifier()
tree.fit(x_train,y_train)
tree_y_pred = tree.predict(x_test)
print(classification_report(y_test,tree_y_pred,target_names=target_name))

               precision    recall  f1-score   support

    High Risk       0.92      0.98      0.95        66
Moderate risk       1.00      1.00      1.00         4
  Low/No Risk       0.96      0.80      0.87        30

     accuracy                           0.93       100
    macro avg       0.96      0.93      0.94       100
 weighted avg       0.93      0.93      0.93       100



In [32]:
# Random forest Model

ranf = RandomForestClassifier()
ranf.fit(x_train,y_train)
ranf_y_pred = ranf.predict(x_test)
print(classification_report(y_test,ranf_y_pred,target_names=target_name))

               precision    recall  f1-score   support

    High Risk       0.96      0.97      0.96        66
Moderate risk       1.00      0.50      0.67         4
  Low/No Risk       0.87      0.90      0.89        30

     accuracy                           0.93       100
    macro avg       0.94      0.79      0.84       100
 weighted avg       0.93      0.93      0.93       100



In [33]:
# XGBoost Model

xgb = XGBClassifier()
xgb.fit(x_train,y_train)
xgb_y_pred = xgb.predict(x_test)
print(classification_report(y_test,xgb_y_pred,target_names=target_name))

               precision    recall  f1-score   support

    High Risk       0.96      0.97      0.96        66
Moderate risk       1.00      0.75      0.86         4
  Low/No Risk       0.90      0.90      0.90        30

     accuracy                           0.94       100
    macro avg       0.95      0.87      0.91       100
 weighted avg       0.94      0.94      0.94       100



## **Result**

**The Logistic Regression Model showed the best performance in predicting the risk of lassa fever among households based on environmental factors**

**LassaGuard AI would therefore be built upon this model**

In [35]:
# Saving the best performing model

joblib.dump(log,"LassaGuard_AI.pkl")

['LassaGuard_AI.pkl']

In [41]:
joblib.dump(ord_enc,"OrdinalEncoder.pkl")

['OrdinalEncoder.pkl']

In [42]:
joblib.dump(one_hot,"OneHotEncoder.pkl")

['OneHotEncoder.pkl']